# Aula 6 — Análise Semântica no PLN

Nesta aula prática, o foco é **análise semântica aplicada**, articulando três níveis complementares:

- 🧩 **Análise sintática (dependências)**: evidencia a estrutura da frase (núcleo verbal, sujeito, complementos).
- 🔎 **Reconhecimento de Entidades Nomeadas (NER)**: identifica menções a objetos relevantes do mundo (pessoas, organizações, locais, datas etc.).
- 🧠 **Análise semântica (aplicada)**: interpreta **papéis**, **relações** e **eventos** a partir de evidências sintáticas e contextuais.

**Importante:** esta aula não aborda Descoberta de Conhecimento em Textos (KDT) nem construção de grafos/bases de conhecimento. A formalização dessas informações será tratada em aula posterior.


## 0. Preparação do Ambiente

Nesta etapa, carregamos o modelo linguístico do spaCy para português e definimos alguns textos de exemplo.

📌 **Observação didática:** o modelo fornece, de forma automática, anotações linguísticas como:
- tokenização,
- classes gramaticais (POS),
- dependências sintáticas,
- entidades nomeadas (NER).

Essas anotações serão utilizadas como **evidências** para a interpretação semântica nas seções seguintes.


In [ ]:
# Instalação da biblioteca spaCy
!pip -q install spacy

In [1]:
import spacy
from spacy.cli import download
from spacy.util import is_package

MODELOS = ["pt_core_news_lg", "pt_core_news_md", "pt_core_news_sm"]

def carregar_modelo():
    ultimo_erro = None
    for m in MODELOS:
        try:
            if not is_package(m):
                # Requer internet no ambiente. Se não houver, instale previamente.
                download(m)
            nlp = spacy.load(m)
            print(f"✅ Modelo carregado: {m}")
            return nlp
        except Exception as e:
            ultimo_erro = e
            print(f"⚠️ Falha ao carregar {m}: {e}")
    raise RuntimeError("Nenhum modelo spaCy disponível. Instale um modelo pt_core_news_* previamente.") from ultimo_erro

nlp = carregar_modelo()

✔ Download and installation successful
You can now load the package via spacy.load('pt_core_news_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
✅ Modelo carregado: pt_core_news_lg


In [17]:
frases = [
    "A Apple anunciou um novo iPhone em São Paulo no dia 12 de setembro de 2025.",
    "João Silva trabalha na Petrobras desde 2020 e foi promovido em 2024.",
    #"O Ministério da Saúde divulgou uma nota em Brasília após o aumento de casos de dengue.",
    #"A Microsoft adquiriu a empresa X por 10 bilhões de dólares em 2023.",
]

### 1. Função para mapear rótulos POS (Universal POS Tags)

O spaCy representa a **classe gramatical** de cada token em `tok.pos_` usando as *Universal POS Tags* (por exemplo: `NOUN`, `PROPN`, `VERB`, `DET`).

Para tornar a leitura mais didática, definimos uma função que converte a sigla em um nome completo em português (por exemplo: `PROPN → substantivo próprio`).


In [7]:
ROTULOS_POS = {
    "ADJ": "adjetivo",
    "ADP": "adposição (preposição)",
    "ADV": "advérbio",
    "AUX": "verbo auxiliar",
    "CCONJ": "conjunção coordenativa",
    "DET": "determinante",
    "INTJ": "interjeição",
    "NOUN": "substantivo",
    "NUM": "numeral",
    "PART": "partícula",
    "PRON": "pronome",
    "PROPN": "substantivo próprio",
    "PUNCT": "pontuação",
    "SCONJ": "conjunção subordinativa",
    "SYM": "símbolo",
    "VERB": "verbo",
    "X": "outro / desconhecido"
}

def nome_completo_pos(tag: str) -> str:
    """
    Recebe uma sigla POS (ex.: 'DET', 'PROPN', 'VERB')
    e retorna o nome completo em português.
    """
    return ROTULOS_POS.get(tag, tag.lower())

### 2. Função para mapear rótulos de dependência (Universal Dependencies)

O spaCy representa a **função sintática** de cada token em `tok.dep_` (por exemplo: `nsubj`, `obj`, `obl`) seguindo convenções do padrão *Universal Dependencies*.

Para facilitar a interpretação em sala, definimos uma função que converte esses rótulos em descrições linguísticas em português (por exemplo: `nsubj → sujeito`).


In [5]:
ROTULOS_DEP = {
    "ROOT": "núcleo da frase",
    "nsubj": "sujeito",
    "nsubj:pass": "sujeito (voz passiva)",
    "obj": "objeto direto",
    "iobj": "objeto indireto",
    "obl": "complemento oblíquo",
    "det": "determinante",
    "amod": "modificador adjetival",
    "advmod": "modificador adverbial",
    "aux": "verbo auxiliar",
    "aux:pass": "verbo auxiliar (voz passiva)",
    "cop": "verbo de ligação",
    "case": "marcador de caso (preposição)",
    "compound": "elemento composto",
    "conj": "conjunto (coordenação)",
    "cc": "conector coordenativo",
    "mark": "marcador subordinativo",
    "nmod": "modificador nominal",
    "appos": "aposição",
    "acl": "oração adjetiva",
    "acl:relcl": "oração relativa",
    "xcomp": "complemento aberto",
    "ccomp": "complemento oracional",
    "punct": "pontuação",
    "fixed": "expressão fixa",
    "flat": "expressão plana",
    "parataxis": "parataxe",
    "expl": "expletivo",
    "discourse": "marcador discursivo"
}

def nome_completo_dep(dep: str) -> str:
    """
    Recebe um rótulo de dependência (ex.: 'nsubj', 'obj', 'obl')
    e retorna uma descrição linguística em português.
    """
    return ROTULOS_DEP.get(dep, dep)

## 3. Análise Sintática (Dependências)

A análise sintática por dependências fornece uma **base estrutural** para a interpretação: indica qual é o núcleo da frase (geralmente um verbo) e como os demais termos se conectam a ele.

Nesta seção, inspecionamos:
- o token,
- sua dependência (`tok.dep_`),
- sua cabeça (head),
- sua classe gramatical (`tok.pos_`).

📌 O objetivo aqui é usar a estrutura para apoiar leituras semânticas posteriores (por exemplo, identificar quem faz o quê e com quem).


**Morfologia e classes gramaticais**  

- **Classe gramatical (POS)** indica a categoria do token (substantivo, verbo, determinante etc.) e é acessada em `tok.pos_`.  
- **Morfologia** refere-se a traços flexionais (por exemplo: gênero, número, tempo verbal) e pode ser acessada em `tok.morph`.

Nesta aula, o foco principal é a **estrutura sintática (dependências)** e como ela sustenta interpretações semânticas. Ainda assim, POS e traços morfológicos podem ajudar a explicar ambiguidades e escolhas do modelo.


In [18]:
def imprimir_dependencias(doc):
    print("Frase:", doc.text)
    print(f"\n{"Token":15} | {"Dependência":25} | {"Cabeça":15} | Classe gramatical")
    print("-" * 80)
    for tok in doc:
      dep_legivel = nome_completo_dep(tok.dep_)
      pos_legivel = nome_completo_pos(tok.pos_)
      print(f"{tok.text:15} | {dep_legivel:25} | {tok.head.text:15} | {pos_legivel}")
    print("-" * 80)

# Executa análise sintática para os textos
docs = [nlp(t) for t in frases]
for doc in docs:
    imprimir_dependencias(doc)


Frase: A Apple anunciou um novo iPhone em São Paulo no dia 12 de setembro de 2025.

Token           | Dependência               | Cabeça          | Classe gramatical
--------------------------------------------------------------------------------
A               | determinante              | Apple           | determinante
Apple           | sujeito                   | anunciou        | substantivo próprio
anunciou        | núcleo da frase           | anunciou        | verbo
um              | determinante              | iPhone          | determinante
novo            | modificador adjetival     | iPhone          | adjetivo
iPhone          | objeto direto             | anunciou        | substantivo
em              | marcador de caso (preposição) | São             | adposição (preposição)
São             | complemento oblíquo       | anunciou        | substantivo próprio
Paulo           | flat:name                 | São             | substantivo próprio
no              | marcador de caso (p

## 4. Reconhecimento de Entidades Nomeadas (NER)

O NER identifica **entidades semanticamente relevantes** no texto (pessoas, organizações, locais, datas, valores etc.).

Para fins didáticos, exibimos rótulos em português (em vez de siglas como `ORG`, `GPE`, `DATE`), de modo que a saída seja lida como uma **classificação semântica** e não apenas como um detalhe técnico do modelo.


In [9]:
ROTULOS_NER = {
    "PERSON": "pessoa",
    "PER": "pessoa",
    "ORG": "organização",
    "GPE": "localização (entidade geopolítica)",
    "LOC": "localização",
    "DATE": "data",
    "TIME": "tempo",
    "MONEY": "valor monetário",
    "PERCENT": "porcentagem",
    "QUANTITY": "quantidade",
    "CARDINAL": "número",
    "ORDINAL": "ordinal",
    "FAC": "instalação",
    "EVENT": "evento",
    "PRODUCT": "produto",
    "WORK_OF_ART": "obra",
    "LANGUAGE": "idioma",
    "MISC": "outros (miscelânea)"
}

def nome_completo_rotulo(label: str) -> str:
    """Retorna um nome legível para o rótulo NER; fallback para o próprio label."""
    return ROTULOS_NER.get(label, label.lower())

def mostrar_entidades(doc):
    print("Texto:", doc.text)
    print("\nEntidades identificadas:")
    if not doc.ents:
        print(" - (nenhuma entidade identificada)")
    for ent in doc.ents:
        print(f" - {ent.text!r:30} | {nome_completo_rotulo(ent.label_):30}")
    print("-" * 80)

for doc in docs:
    mostrar_entidades(doc)


Texto: A Apple anunciou um novo iPhone em São Paulo no dia 12 de setembro de 2025.

Entidades identificadas:
 - 'Apple'                        | organização                   
 - 'iPhone'                       | outros (miscelânea)           
 - 'São Paulo'                    | localização                   
--------------------------------------------------------------------------------
Texto: João Silva trabalha na Petrobras desde 2020 e foi promovido em 2024.

Entidades identificadas:
 - 'João Silva'                   | pessoa                        
 - 'Petrobras'                    | organização                   
--------------------------------------------------------------------------------
Texto: O Ministério da Saúde divulgou uma nota em Brasília após o aumento de casos de dengue.

Entidades identificadas:
 - 'Ministério da Saúde'          | localização                   
 - 'Brasília'                     | localização                   
--------------------------------------

## 5. Análise Semântica (Aplicada)

Nesta seção, combinamos **NER + dependências** para produzir uma interpretação semântica mínima, baseada em:

- 🎭 **Papéis** (por exemplo: *agente* e *alvo*, aproximados a partir de sujeito/objeto),
- 🔗 **Relações** (por exemplo: *sujeito —verbo→ objeto*),
- ⏱️ **Eventos** (verbo como gatilho + participantes + tempo/local quando disponível).

📌 O objetivo é **interpretar o texto** e discutir limites/ambiguidade dos resultados — **não** formalizar essas saídas como base de conhecimento nesta etapa.


In [19]:
from dataclasses import dataclass
from typing import List, Tuple, Optional

@dataclass
class Relacao:
    sujeito: str
    relacao: str
    objeto: str
    evidencia: str

@dataclass
class Evento:
    gatilho: str
    participantes: List[Tuple[str, str]]  # (papel, texto)
    tempo: Optional[str]
    local: Optional[str]
    evidencia: str

def _texto_span_entidade(doc, token):
    # Se o token estiver dentro de uma entidade, retorna o texto da entidade
    for ent in doc.ents:
        if ent.start <= token.i < ent.end:
            return ent.text
    return token.text

def extrair_relacoes(doc) -> List[Relacao]:
    rels: List[Relacao] = []
    for tok in doc:
        if tok.pos_ not in ("VERB", "AUX"):
            continue

        sujeitos = [c for c in tok.children if c.dep_ in ("nsubj", "nsubj:pass")]
        objetos  = [c for c in tok.children if c.dep_ in ("obj", "iobj", "obl")]

        if not sujeitos or not objetos:
            continue

        for s in sujeitos:
            subj_txt = _texto_span_entidade(doc, s)
            for o in objetos:
                obj_txt = _texto_span_entidade(doc, o)

                # filtro didático: exige alguma entidade envolvida (reduz ruído)
                subj_eh_ent = any(ent.text == subj_txt for ent in doc.ents)
                obj_eh_ent  = any(ent.text == obj_txt for ent in doc.ents)
                if not (subj_eh_ent or obj_eh_ent):
                    continue

                rels.append(Relacao(subj_txt, tok.lemma_, obj_txt, doc.text))
    return rels

def extrair_eventos(doc) -> List[Evento]:
    eventos: List[Evento] = []
    tempos = [ent.text for ent in doc.ents if ent.label_ in ("DATE", "TIME")]
    locais = [ent.text for ent in doc.ents if ent.label_ in ("GPE", "LOC")]

    for tok in doc:
        if tok.pos_ not in ("VERB", "AUX"):
            continue

        sujeitos = [c for c in tok.children if c.dep_ in ("nsubj", "nsubj:pass")]
        objetos  = [c for c in tok.children if c.dep_ in ("obj", "iobj", "obl")]

        participantes: List[Tuple[str, str]] = []
        for s in sujeitos:
            participantes.append(("AGENTE", _texto_span_entidade(doc, s)))
        for o in objetos:
            participantes.append(("ALVO", _texto_span_entidade(doc, o)))

        if not participantes:
            continue

        eventos.append(Evento(
            gatilho=tok.lemma_,
            participantes=participantes,
            tempo=tempos[0] if tempos else None,
            local=locais[0] if locais else None,
            evidencia=doc.text
        ))
    return eventos

def relatorio_semantico(doc):
    print("\nTEXTO ANALISADO:")
    print(doc.text)

    print("\n[ENTIDADES]")
    if not doc.ents:
        print(" - (nenhuma)")
    for ent in doc.ents:
        print(f" - {ent.text} ({nome_completo_rotulo(ent.label_)})")

    print("\n[RELAÇÕES (Sujeito —verbo→ Objeto)]")
    rels = extrair_relacoes(doc)
    if not rels:
        print(" - (nenhuma relação explícita identificada)")
    else:
        for r in rels:
            print(f" - {r.sujeito} --{r.relacao}--> {r.objeto}")

    print("\n[EVENTOS (gatilho + participantes + tempo/local)]")
    evs = extrair_eventos(doc)
    if not evs:
        print(" - (nenhum evento identificado)")
    else:
        for e in evs:
            parts = ", ".join([f"{papel}:{ent}" for papel, ent in e.participantes])
            print(f" - Evento '{e.gatilho}' | tempo={e.tempo or 'não identificado'} | local={e.local or 'não identificado'} | {parts}")

    print("=" * 90)

for doc in docs:
    relatorio_semantico(doc)



TEXTO ANALISADO:
A Apple anunciou um novo iPhone em São Paulo no dia 12 de setembro de 2025.

[ENTIDADES]
 - Apple (organização)
 - iPhone (outros (miscelânea))
 - São Paulo (localização)

[RELAÇÕES (Sujeito —verbo→ Objeto)]
 - Apple --anunciar--> iPhone
 - Apple --anunciar--> São Paulo
 - Apple --anunciar--> dia

[EVENTOS (gatilho + participantes + tempo/local)]
 - Evento 'anunciar' | tempo=não identificado | local=São Paulo | AGENTE:Apple, ALVO:iPhone, ALVO:São Paulo, ALVO:dia

TEXTO ANALISADO:
João Silva trabalha na Petrobras desde 2020 e foi promovido em 2024.

[ENTIDADES]
 - João Silva (pessoa)
 - Petrobras (organização)

[RELAÇÕES (Sujeito —verbo→ Objeto)]
 - João Silva --trabalhar--> Petrobras
 - João Silva --trabalhar--> 2020

[EVENTOS (gatilho + participantes + tempo/local)]
 - Evento 'trabalhar' | tempo=não identificado | local=não identificado | AGENTE:João Silva, ALVO:Petrobras, ALVO:2020
 - Evento 'promover' | tempo=não identificado | local=não identificado | ALVO:2024


## Exercício 1 – Interpretação de datas
Enunciado

No relatório semântico gerado pelo notebook, observa-se que expressões temporais nem sempre são interpretadas corretamente. Datas completas, como “12 de setembro de 2025”, podem não ser reconhecidas como uma única entidade temporal, enquanto anos isolados, como “2020” e “2024”, deixam de ser identificados como datas. Como consequência, o campo tempo dos eventos frequentemente permanece como “não identificado”, e partes da expressão temporal aparecem como objetos ou participantes indevidos (por exemplo, “dia”).

Sem modificar nenhuma das funções já implementadas (`extrair_relacoes`, `extrair_eventos`, `relatorio_semantico` etc.), adicione novas células ao notebook para:

1.  Identificar corretamente expressões de data completas, como “12 de setembro de 2025”, evitando que seus componentes apareçam de forma fragmentada;

2.  Reconhecer anos isolados (por exemplo, 2020 e 2024) como entidades temporais;

3.  Garantir que todas essas expressões sejam reconhecidas como entidades do tipo DATE;

4.  Permitir que o campo tempo dos eventos seja preenchido adequadamente a partir dessas entidades.

Utilize exclusivamente recursos do pipeline do spaCy, como regras, padrões ou componentes adicionais, sem alterar o código já fornecido.

In [33]:
# ------------------------
# Resposta do Exercício 1
# ------------------------

from spacy.pipeline import EntityRuler

# Verifica se o 'entity_ruler' já existe na pipeline
if "entity_ruler" not in nlp.pipe_names:
    ruler = nlp.add_pipe("entity_ruler", before="ner")
else:
    # Se já existir, obtém a instância existente
    ruler = nlp.get_pipe("entity_ruler")

padroes_datas = [
    {
        "label": "DATE",
        "pattern": [
            {"LOWER": "dia"},
            {"LIKE_NUM": True},
            {"LOWER": "de"},
            {"LOWER": {"IN": [
                "janeiro", "fevereiro", "março", "abril", "maio", "junho",
                "julho", "agosto", "setembro", "outubro", "novembro", "dezembro"
            ]}},
            {"LOWER": "de"},
            {"LIKE_NUM": True}
        ]
    },
    {
        "label": "DATE",
        "pattern": [
            {"SHAPE": "dddd"}  # 4 dígitos
        ]
    }
]

ruler.add_patterns(padroes_datas)

In [34]:
docs = [nlp(texto) for texto in frases]

for doc in docs:
    relatorio_semantico(doc)


TEXTO ANALISADO:
A Apple anunciou um novo iPhone em São Paulo no dia 12 de setembro de 2025.

[ENTIDADES]
 - Apple (organização)
 - iPhone (outros (miscelânea))
 - São Paulo (localização)
 - dia 12 de setembro de 2025 (data)

[RELAÇÕES (Sujeito —verbo→ Objeto)]
 - Apple --anunciar--> iPhone
 - Apple --anunciar--> São Paulo
 - Apple --anunciar--> dia 12 de setembro de 2025

[EVENTOS (gatilho + participantes + tempo/local)]
 - Evento 'anunciar' | tempo=dia 12 de setembro de 2025 | local=São Paulo | AGENTE:Apple, ALVO:iPhone, ALVO:São Paulo, ALVO:dia 12 de setembro de 2025

TEXTO ANALISADO:
João Silva trabalha na Petrobras desde 2020 e foi promovido em 2024.

[ENTIDADES]
 - João Silva (pessoa)
 - Petrobras (organização)
 - 2020 (data)
 - 2024 (data)

[RELAÇÕES (Sujeito —verbo→ Objeto)]
 - João Silva --trabalhar--> Petrobras
 - João Silva --trabalhar--> 2020

[EVENTOS (gatilho + participantes + tempo/local)]
 - Evento 'trabalhar' | tempo=2020 | local=não identificado | AGENTE:João Silva, 